In [18]:
# 1. Clone repo training chuẩn (Đây là backend của EasyOCR)
!git clone https://github.com/clovaai/deep-text-recognition-benchmark.git

# 2. Cài đặt các thư viện cần thiết
!pip install -q fire lmdb opencv-python nltk natsort
!pip install -q huggingface_hub  # Để tải data từ HF

print("✅ Đã setup xong môi trường!")

fatal: destination path 'deep-text-recognition-benchmark' already exists and is not an empty directory.
✅ Đã setup xong môi trường!


In [19]:
import os
from datasets import load_dataset
from tqdm import tqdm
import cv2
import numpy as np

# 1. Cấu hình
DATASET_ID = "vklinhhh/OCR_handwritting_HAT2023"
RAW_IMG_DIR = "/kaggle/working/raw_images"
OUTPUT_LMDB = "/kaggle/working/data_lmdb"

os.makedirs(RAW_IMG_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_LMDB}/train", exist_ok=True)
os.makedirs(f"{OUTPUT_LMDB}/valid", exist_ok=True)

# 2. Tải Dataset
print(f"--> Đang tải dataset {DATASET_ID}...")
dataset = load_dataset(DATASET_ID, split="train")

# In thử mẫu đầu tiên để check cột label
print("Mẫu dữ liệu đầu tiên:", dataset[0])

# Chia train/valid
dataset = dataset.train_test_split(test_size=0.1)

def process_data(data_split, split_name):
    gt_file_path = f"gt_{split_name}.txt"
    print(f"--> Đang xử lý tập {split_name}...")
    
    with open(gt_file_path, 'w', encoding='utf-8') as f:
        for idx, item in enumerate(tqdm(data_split)):
            try:
                image = item['image']
                # Lấy label (thử các key phổ biến)
                label = item.get('handwriting') or item.get('text') or item.get('label')
                
                if label is None: continue
                
                # Filter: Nếu label quá dài (>20 ký tự) có thể bỏ qua để tránh lỗi OOM 
                # nhưng với SBD thì cứ giữ để model học nét chữ
                
                img_name = f"{split_name}_{idx}.jpg"
                img_path = os.path.join(RAW_IMG_DIR, img_name)
                image.save(img_path)
                f.write(f"{img_name}\t{label}\n")
            except Exception as e:
                pass # Bỏ qua lỗi nhỏ
    return gt_file_path

train_gt = process_data(dataset['train'], 'train')
valid_gt = process_data(dataset['test'], 'valid')

print("--> Đang tạo LMDB...")
!python deep-text-recognition-benchmark/create_lmdb_dataset.py --inputPath {RAW_IMG_DIR} --gtFile {train_gt} --outputPath {OUTPUT_LMDB}/train
!python deep-text-recognition-benchmark/create_lmdb_dataset.py --inputPath {RAW_IMG_DIR} --gtFile {valid_gt} --outputPath {OUTPUT_LMDB}/valid

print("✅ ĐÃ XONG LMDB!")

--> Đang tải dataset vklinhhh/OCR_handwritting_HAT2023...
Mẫu dữ liệu đầu tiên: {'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=93x70 at 0x7D58819F8260>, 'label': 'OĂM'}
--> Đang xử lý tập train...


100%|██████████| 74160/74160 [01:07<00:00, 1105.84it/s]


--> Đang xử lý tập valid...


100%|██████████| 8240/8240 [00:07<00:00, 1073.68it/s]


--> Đang tạo LMDB...
Written 1000 / 74160
Written 2000 / 74160
Written 3000 / 74160
Written 4000 / 74160
Written 5000 / 74160
Written 6000 / 74160
Written 7000 / 74160
Written 8000 / 74160
Written 9000 / 74160
Written 10000 / 74160
Written 11000 / 74160
Written 12000 / 74160
Written 13000 / 74160
Written 14000 / 74160
Written 15000 / 74160
Written 16000 / 74160
Written 17000 / 74160
Written 18000 / 74160
Written 19000 / 74160
Written 20000 / 74160
Written 21000 / 74160
Written 22000 / 74160
Written 23000 / 74160
Written 24000 / 74160
Written 25000 / 74160
Written 26000 / 74160
Written 27000 / 74160
Written 28000 / 74160
Written 29000 / 74160
Written 30000 / 74160
Written 31000 / 74160
Written 32000 / 74160
Written 33000 / 74160
Written 34000 / 74160
Written 35000 / 74160
Written 36000 / 74160
Written 37000 / 74160
Written 38000 / 74160
Written 39000 / 74160
Written 40000 / 74160
Written 41000 / 74160
Written 42000 / 74160
Written 43000 / 74160
Written 44000 / 74160
Written 45000 / 7416

In [20]:
import os
import requests

# Đường dẫn file
model_path = "/kaggle/working/pre_trained.pth"

# Xóa file cũ
if os.path.exists(model_path):
    os.remove(model_path)

# Link tải latin.pth (Ổn định hơn english_g2)
url = "https://huggingface.co/spaces/Ajay-user/Optical-Character-Recognition/resolve/main/EasyOCR/model/english_g2.pth"

print(f"⬇️ Đang thử tải latin.pth từ: {url}...")

# Dùng requests giả lập trình duyệt để tránh bị chặn
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

try:
    response = requests.get(url, headers=headers, stream=True)
    if response.status_code == 200:
        with open(model_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"✅ Tải thành công! Dung lượng: {os.path.getsize(model_path)/1024/1024:.2f} MB")
    else:
        print(f"❌ Lỗi HTTP: {response.status_code}")
except Exception as e:
    print(f"❌ Lỗi kết nối: {e}")

# Kiểm tra cuối cùng
if os.path.exists(model_path) and os.path.getsize(model_path) > 1000000:
    print("🎉 SẴN SÀNG TRAIN!")
else:
    print("⛔ Vẫn không tải được. Hãy dùng GIẢI PHÁP 1 (Add Input) ở trên nhé!")

⬇️ Đang thử tải latin.pth từ: https://huggingface.co/spaces/Ajay-user/Optical-Character-Recognition/resolve/main/EasyOCR/model/english_g2.pth...
✅ Tải thành công! Dung lượng: 14.44 MB
🎉 SẴN SÀNG TRAIN!


In [21]:
# --- FIX LỖI Import torch._utils._accumulate ---
# Lỗi này do repo cũ không tương thích PyTorch mới.
# Ta thay thế dòng import lỗi bằng hàm accumulate của thư viện chuẩn itertools (chức năng y hệt).

file_path = "/kaggle/working/deep-text-recognition-benchmark/dataset.py"

# Đọc nội dung file
with open(file_path, "r") as f:
    content = f.read()

# Thay thế dòng import lỗi
# Code cũ: from torch._utils import _accumulate
# Code mới: from itertools import accumulate as _accumulate
content = content.replace("from torch._utils import _accumulate", "from itertools import accumulate as _accumulate")

# Ghi lại file
with open(file_path, "w") as f:
    f.write(content)

print("✅ Đã vá lỗi '_accumulate' trong dataset.py thành công!")

✅ Đã vá lỗi '_accumulate' trong dataset.py thành công!


In [25]:
# --- VÁ LỖI LOAD MODEL (SMART LOADING) ---
# Script này sẽ sửa file train.py để nó tự động BỎ QUA các layer bị lệch size (như Prediction)
# thay vì báo lỗi và dừng chương trình.

train_py_path = "/kaggle/working/deep-text-recognition-benchmark/train.py"

with open(train_py_path, "r") as f:
    content = f.read()

# Đoạn code gốc gây lỗi
old_code = "model.load_state_dict(torch.load(opt.saved_model), strict=False)"

# Đoạn code mới (Lọc bỏ các key lệch size trước khi load)
new_code = """
    print("--> Đang thực hiện Smart Loading weights...")
    state_dict = torch.load(opt.saved_model)
    model_dict = model.state_dict()
    # Lọc bỏ các key không khớp kích thước (ví dụ: Prediction layer)
    new_state_dict = {k: v for k, v in state_dict.items() if k in model_dict and v.shape == model_dict[k].shape}
    for k, v in state_dict.items():
        if k not in new_state_dict:
            print(f"    Bỏ qua layer lệch size: {k} (Shape cũ: {v.shape})")
    model.load_state_dict(new_state_dict, strict=False)
"""

if old_code in content:
    content = content.replace(old_code, new_code)
    with open(train_py_path, "w") as f:
        f.write(content)
    print("✅ Đã vá lỗi Smart Loading thành công!")
else:
    print("⚠️ Không tìm thấy đoạn code cần sửa (Có thể đã sửa rồi).")

✅ Đã vá lỗi Smart Loading thành công!


In [27]:
import os

# --- BƯỚC 1: RESET LẠI CODE (Xóa cái cũ bị lỗi đi làm lại cho sạch) ---
print("🧹 Đang dọn dẹp thư mục cũ...")
!rm -rf deep-text-recognition-benchmark
!git clone https://github.com/clovaai/deep-text-recognition-benchmark.git
print("✅ Đã Clone lại repo mới.")

# --- BƯỚC 2: VÁ LỖI 'accumulate' (Do PyTorch mới đổi tên hàm) ---
dataset_path = "deep-text-recognition-benchmark/dataset.py"
with open(dataset_path, "r") as f:
    content = f.read()
content = content.replace("from torch._utils import _accumulate", "from itertools import accumulate as _accumulate")
with open(dataset_path, "w") as f:
    f.write(content)
print("✅ Đã vá lỗi import accumulate.")

# --- BƯỚC 3: VÁ LỖI 'Smart Loading' (Bỏ qua layer lệch size) ---
# Lần này mình thêm thụt đầu dòng (indent) chuẩn 12 dấu cách để không bị lỗi IndentationError
train_path = "deep-text-recognition-benchmark/train.py"

smart_loading_code = """
            # --- SMART LOADING FIX ---
            print("--> Smart Loading: Đang lọc bỏ các layer lệch size...")
            state_dict = torch.load(opt.saved_model)
            model_dict = model.state_dict()
            # Chỉ lấy các key khớp size
            new_state_dict = {k: v for k, v in state_dict.items() if k in model_dict and v.shape == model_dict[k].shape}
            model.load_state_dict(new_state_dict, strict=False)
            # -------------------------
"""

with open(train_path, "r") as f:
    lines = f.readlines()

new_lines = []
target_line = "model.load_state_dict(torch.load(opt.saved_model), strict=False)"

for line in lines:
    if target_line in line:
        # Thay thế dòng gây lỗi bằng block code mới
        new_lines.append(smart_loading_code)
    else:
        new_lines.append(line)

with open(train_path, "w") as f:
    f.writelines(new_lines)

print("✅ Đã vá lỗi Smart Loading (với Indentation chuẩn).")
print("\n🎉 MỌI THỨ ĐÃ SẴN SÀNG! BẠN HÃY CHẠY LẠI CELL TRAIN (CELL 5).")

🧹 Đang dọn dẹp thư mục cũ...
Cloning into 'deep-text-recognition-benchmark'...
remote: Enumerating objects: 499, done.
remote: Counting objects: 100% (225/225), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 499 (delta 208), reused 200 (delta 200), pack-reused 274 (from 1)
Receiving objects: 100% (499/499), 3.05 MiB | 17.18 MiB/s, done.
Resolving deltas: 100% (308/308), done.
✅ Đã Clone lại repo mới.
✅ Đã vá lỗi import accumulate.
✅ Đã vá lỗi Smart Loading (với Indentation chuẩn).

🎉 MỌI THỨ ĐÃ SẴN SÀNG! BẠN HÃY CHẠY LẠI CELL TRAIN (CELL 5).


In [48]:
# --- CELL 5: TRAIN TIẾP (RESUME) ---

VN_CHARS = "0123456789abcdefghijklmnopqrstuvwxyzáàảãạăắằẳẵặâấầẩẫậđéèẻẽẹêếềểễệíìỉĩịóòỏõọôốồổỗộơớờởỡợúùủũụưứừửữựýỳỷỹỵABCDEFGHIJKLMNOPQRSTUVWXYZÁÀẢÃẠĂẮẰẲẴẶÂẤẦẨẪẬĐÉÈẺẼẸÊẾỀỂỄỆÍÌỈĨỊÓÒỎÕỌÔỐỒỔỖỘƠỚỜỞỠỢÚÙỦŨỤƯỨỪỬỮỰÝỲỶỸỴ"

# --- LỆNH TRAIN TIẾP (RESUME) ---
!python deep-text-recognition-benchmark/train.py \
    --train_data /kaggle/working/data_lmdb/train \
    --valid_data /kaggle/working/data_lmdb/valid \
    --select_data / \
    --batch_ratio 1.0 \
    --Transformation None \
    --FeatureExtraction ResNet \
    --SequenceModeling BiLSTM \
    --Prediction CTC \
    --saved_model /kaggle/working/saved_models/my_sbd_model_resume_v2/best_accuracy.pth \
    --FT \
    --character "{VN_CHARS}" \
    --num_iter 1000 \
    --valInterval 200 \
    --batch_size 64 \
    --workers 0 \
    --lr 0.1 \
    --output_channel 256 \
    --hidden_size 256 \
    --exp_name my_sbd_model_resume_v3

------ Use multi-GPU setting ------
if you stuck too long time with multi-GPU setting, try to set --workers 0
Filtering the images containing characters which are not in opt.character
Filtering the images whose label is longer than opt.batch_max_length
--------------------------------------------------------------------------------
dataset_root: /kaggle/working/data_lmdb/train
opt.select_data: ['/']
opt.batch_ratio: ['1.0']
--------------------------------------------------------------------------------
dataset_root:    /kaggle/working/data_lmdb/train	 dataset: /
sub-directory:	/.	 num samples: 68870
num total samples of /: 68870 x 1.0 (total_data_usage_ratio) = 68870
num samples of / per batch: 128 x 1.0 (batch_ratio) = 128
--------------------------------------------------------------------------------
Total_batch_size: 128 = 128
--------------------------------------------------------------------------------
dataset_root:    /kaggle/working/data_lmdb/valid	 dataset: /
sub-directory:

In [29]:
# --- TẢI MODEL TỐT NHẤT VỀ (Đã lưu tại vòng 1200) ---
import os
from IPython.display import FileLink

# Đường dẫn file model tốt nhất
best_model_path = 'saved_models/my_sbd_model/best_accuracy.pth'

if os.path.exists(best_model_path):
    print(f"✅ Tìm thấy model ngon! (Vòng 1200 - Acc: 48.8%)")
    display(FileLink(best_model_path))
else:
    print("❌ Không tìm thấy file. Kiểm tra lại thư mục saved_models.")

✅ Tìm thấy model ngon! (Vòng 1200 - Acc: 48.8%)


/kaggle/working/saved_models/my_sbd_model/best_accuracy.pth